# Difference in differences constructs a counterfactual trend

We observe treated and comparison groups before and after an intervention. Difference in differences subtracts the comparison group’s change from the treated group’s change.

## 1. Using a standard statistical library
Fit the interaction regression with statsmodels; its treated × post coefficient is the difference-in-differences estimate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.formula.api as smf

rng = np.random.default_rng(8)
n, true_effect = 300, 2.5
control_pre = rng.normal(10, 1, n)
control_post = control_pre + 1 + rng.normal(0, .35, n)
treated_pre = rng.normal(11, 1, n)
treated_post = treated_pre + 1 + true_effect + rng.normal(0, .35, n)

frame = pd.DataFrame({
    'outcome': np.r_[control_pre, control_post, treated_pre, treated_post],
    'treated': np.repeat([0, 0, 1, 1], n),
    'post': np.repeat([0, 1, 0, 1], n)
})
model = smf.ols('outcome ~ treated * post', data=frame).fit()
did = model.params['treated:post']
print(f'True effect: {true_effect:.2f}')
print(f'statsmodels estimate: {did:.2f}')

control_change = control_post.mean() - control_pre.mean()
means_control = [control_pre.mean(), control_post.mean()]
means_treated = [treated_pre.mean(), treated_post.mean()]
counterfactual = [treated_pre.mean(), treated_pre.mean() + control_change]
plt.plot(['before', 'after'], means_control, 'o-', label='comparison')
plt.plot(['before', 'after'], means_treated, 'o-', label='treated')
plt.plot(['before', 'after'], counterfactual, 'o--', label='treated counterfactual')
plt.ylabel('outcome'); plt.title('Observed change and constructed counterfactual'); plt.legend(); plt.show();

## 2. Build the estimator from scratch
Compute the four cell means and subtract the comparison group’s change from the treated group’s change.

In [ ]:
control_change = control_post.mean() - control_pre.mean()
treated_change = treated_post.mean() - treated_pre.mean()
did_from_scratch = treated_change - control_change
print(f'From-scratch estimate: {did_from_scratch:.2f}')

## What to notice
The treated counterfactual is never observed. It is constructed by applying the comparison group’s change. The estimate is credible only when the parallel-trends assumption and the rest of the design are credible.